In [ ]:

import os
import requests
import io
import re
import time
import sys
import traceback
import warnings
import pandas as pd
import numpy as np
import gspread
from datetime import datetime, timedelta
from oauth2client.service_account import ServiceAccountCredentials
from googleapiclient.discovery import build
from tqdm import tqdm

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
JSON_PATH = r"C:\Users\muralimohana.s\Desktop\milk_npd\minutesreports-8b030d50afd3.json"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

print("Imports loaded.")

# Sheet IDs
DEST_SPREADSHEET_ID = "1qo_pVL0dH9p4vF8oTCOwb66wDFZvykeJkLHTe6uORKs"
DEST_TAB_NAME = "data"  
IB_LOGS_TAB_NAME = "I/B Logs" 

STORE_MASTER_ID = "1qU3DEy5hhHwyN0p9iokqT5K5j7GK3GM78ta_R4zK2nA"
EVENT_HUB_ID = "1rC6iEbocHDlwNo9NlmOjhOFMnZnw5vHZjETa3ZkrLN0"
STORE_DASH_ID = "1MJc7zRkKSosDZ5NPi_W9mSwixOvt0qtErww6Pg9Fjx0"
TAT_CSV_LINK_ID = "1OyVoywFPWL7hbMEjmS7ZcsM-3buHliC3nuRJZI2aP1c"
RETURNS_SALES_ID = "1irFnXuUy4PdJBtnoh_rVSg7vQ9EZ2w1xM4qK-xldYiw"
LOOKUP_ID = "10JYZdk4WGLvLIJborLPt0d3i3q0Bzo85CzD0hvD6L_A"
STOREWISE_SUMMARY_ID = "1szthmoJgctVD0sdor5X9TMSz6fM4cfQat4BzfOubbPk"
MILK_JOINED_ID = "1O_k3rBaDoyCVAKPQUrBPrjl-lUUxqvBTNgm9nV9bCGA"
INF_DATA_ID = "1T-_tUV4Bx_PduamD04bo6saoZApQbtxzKZoKD2sVXSg"
PIVOT_INF_ID = "1zkVrpDbxNcwn87kcp8TCyIZFVlpdpEsDIP-XZ4YbtMs"
CHILLER_CHECK_ID = "1ucoYLh3NsdZE7i7FfJ-xkcIjNxGWL75wYS_621L3QvQ"
INDENT_ID = "1C7nPOOqIVjZU2iZr6Ou8qkS9Kz5CcbbrOVhT_P2Tr7s"
MAPPING_SHEET_ID = "10JYZdk4WGLvLIJborLPt0d3i3q0Bzo85CzD0hvD6L_A" 
TTL_D_1 = "11v-jZy18AtAawAbWUkiwqtvuBxqZiBwWto8SzcPoLiA"
# Dates
TODAY = datetime.now()
YESTERDAY = (TODAY - timedelta(days=1)).date()
YESTERDAY_STR = str(YESTERDAY)

# Authentication Functions
def auth_gspread():
    creds = ServiceAccountCredentials.from_json_keyfile_name(JSON_PATH, SCOPES)
    return gspread.authorize(creds)

def auth_drive():
    creds = ServiceAccountCredentials.from_json_keyfile_name(JSON_PATH, SCOPES)
    return build('drive', 'v3', credentials=creds)

def download_csv_tqdm(url, desc="Downloading CSV"):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total = int(response.headers.get('content-length', 0))
    
    text = io.BytesIO()
    with tqdm(desc=desc, total=total, unit='iB', unit_scale=True, unit_divisor=1024) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = text.write(data)
            bar.update(len(data))
    text.seek(0)
    return pd.read_csv(text, low_memory=False)

def get_sheet_df(client, ss_id, tab_name=None, header_row=0):
    ss = gspread_retry(lambda: client.open_by_key(ss_id))
    ws = ss.worksheet(tab_name) if tab_name else ss.get_worksheet(0)
    data = gspread_retry(lambda: ws.get_all_values())
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data[header_row+1:], columns=data[header_row])
    return df


import pickle
from gspread.exceptions import APIError

CHECKPOINT_PATH = r"C:\Users\muralimohana.s\Downloads\pipeline_checkpoint.pkl"

def gspread_retry(fn, retries=5, backoff=15):
    """Auto-retry any gspread call on transient 503/500 errors."""
    for attempt in range(retries):
        try:
            return fn()
        except APIError as e:
            code = e.response.status_code if hasattr(e, 'response') else 0
            if attempt < retries - 1 and code in (429, 500, 503):
                wait = backoff * (2 ** attempt)
                print(f"  ⚠️ Google API {code} — retrying in {wait}s (attempt {attempt+1}/{retries})...")
                time.sleep(wait)
            else:
                raise

def save_checkpoint(df):
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(df, f)
    print(f"  💾 Checkpoint saved → {CHECKPOINT_PATH}")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "rb") as f:
            df = pickle.load(f)
        print(f"  ✅ Checkpoint loaded from {CHECKPOINT_PATH} ({len(df)} rows, {len(df.columns)} cols)")
        return df
    return None

print(f"--- Pipeline Ready for D-1 Date: {YESTERDAY_STR} ---")
client = auth_gspread()
drive_service = auth_drive()

# %%
# 0. FRESH MILK FSN MAPPING (used to identify Milk records by FSN across all sections)
print("Fetching Fresh Milk FSN mapping...")
freshmilk_ws = gspread_retry(lambda: client.open_by_key(MAPPING_SHEET_ID)).worksheet("freshmilk_fsn")
freshmilk_fsn_set = {str(v).strip() for v in gspread_retry(lambda: freshmilk_ws.col_values(2))[1:] if str(v).strip()}

# ==============================================================
# RESUME FROM CHECKPOINT (optional - skip to any step)
# If the pipeline crashed mid-way and final_df is in memory, ignore this.
# If you restarted the kernel, uncomment the line below and run ONLY the
# setup block above + the step you want to resume from.
#
#   final_df = load_checkpoint()
#
# ==============================================================
# %%
# 1. STORE DETAILS (Base DataFrame)
print("Fetching Store Details...")
df_stores = get_sheet_df(client, STORE_MASTER_ID, "Store_details")
df_stores = df_stores[['Store Code', 'Zone', 'City.Final']].drop_duplicates(subset=['Store Code']).copy()
df_stores.rename(columns={'Store Code': 'store_id'}, inplace=True)

# STANDARD LOWERCASE CLEANING
df_stores['store_id'] = df_stores['store_id'].astype(str).str.strip().str.lower()

final_df = df_stores.copy()
final_df['Date'] = YESTERDAY_STR
print("Step 1 Complete.")
save_checkpoint(final_df)

# %%
# 2. EVENT HUB
print("Fetching Event Hub Data...")
df_event = get_sheet_df(client, EVENT_HUB_ID,"Sheet1")
if not df_event.empty:
    df_event['OrderDate'] = pd.to_datetime(df_event['OrderDate'], errors='coerce').dt.date
    df_event = df_event[(df_event['OrderDate'] == YESTERDAY) & (df_event['Flag'].str.lower() != 'mnow')].copy()
    
    # STANDARD LOWERCASE CLEANING
    df_event['Event Hub Code'] = df_event['Event Hub Code'].astype(str).str.strip().str.lower()
    
    # --- FIX: Remove commas and force to numbers BEFORE summing ---
    for col in ['Total Orders_sum', 'Num Pieces_sum', 'p_RTO']:
        if col in df_event.columns:
            df_event[col] = pd.to_numeric(df_event[col].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    
    agg_event = df_event.groupby('Event Hub Code').agg({
        'Total Orders_sum': 'sum',
        'Num Pieces_sum': 'sum',
        'p_RTO': 'sum'
    }).reset_index().rename(columns={'Event Hub Code': 'store_id'})
    
    final_df = pd.merge(final_df, agg_event, on='store_id', how='left')
print("Step 2 Complete.")

# %%
# 3. STORE DASHBOARD (Manpower & Utilization)
print("Fetching Store Dashboard...")
df_dash = get_sheet_df(client, STORE_DASH_ID, "Imp_CX_data", header_row=0)
if not df_dash.empty and 'DS ID' in df_dash.columns:
    df_dash = df_dash[['DS ID', 'AOP', 'Actual',"UPL (D-3)", 'Dry Utilisation', 'Freezer Utilisation', 'Chiller Utilisation',"Landed qty (D-1)","Planned Creations","Putaway completed (6AM) D-1","on time putaway qty"]].copy()
    df_dash.rename(columns={'DS ID': 'store_id', 'AOP': 'Manpower - Planned count', 'Actual': 'Manpower - Actual count'}, inplace=True)
    
    # STANDARD LOWERCASE CLEANING
    df_dash['store_id'] = df_dash['store_id'].astype(str).str.strip().str.lower()
    
    df_dash = df_dash.drop_duplicates(subset=['store_id'])
    final_df = pd.merge(final_df, df_dash, on='store_id', how='left')
print("Step 3 Complete.")

# %%
# 4. TAT CSV DOWNLOAD
print("Fetching TAT Data...")
df_tat_link = get_sheet_df(client, TAT_CSV_LINK_ID)
if not df_tat_link.empty:
    df_tat_link['TimeCol'] = pd.to_datetime(df_tat_link.iloc[:, 0], errors='coerce')
    df_tat_link = df_tat_link[df_tat_link['TimeCol'].dt.date == YESTERDAY]
    if not df_tat_link.empty:
        latest_url = df_tat_link.sort_values('TimeCol', ascending=False).iloc[0, 2] 
        df_tat = download_csv_tqdm(latest_url, desc="Downloading TAT CSV")
        
        # STANDARD LOWERCASE CLEANING
        df_tat['warehouse_name'] = df_tat['warehouse_name'].astype(str).str.strip().str.lower()
        
        # FIX: Convert the "0 Days, 00:01:40" string into total seconds
        df_tat['pc2pa_avg_tat_sec'] = pd.to_timedelta(df_tat['pc2pa_avg_tat_sec'], errors='coerce').dt.total_seconds().fillna(0)
        df_tat['ps2d_avg_tat_sec'] = pd.to_timedelta(df_tat['ps2d_avg_tat_sec'], errors='coerce').dt.total_seconds().fillna(0)
        
        tat_agg = df_tat.groupby('warehouse_name')[['pc2pa_avg_tat_sec', 'ps2d_avg_tat_sec']].sum().reset_index()
        tat_agg['pc2pa_avg_tat_sec'] = tat_agg['pc2pa_avg_tat_sec'] / 60.0
        tat_agg['ps2d_avg_tat_sec'] = tat_agg['ps2d_avg_tat_sec'] / 60.0
        tat_agg.rename(columns={'warehouse_name': 'store_id', 'pc2pa_avg_tat_sec': 'pc2pa_avg_tat_min', 'ps2d_avg_tat_sec': 'ps2d_avg_tat_min'}, inplace=True)

        final_df = pd.merge(final_df, tat_agg, on='store_id', how='left')
print("Step 4 Complete.")
save_checkpoint(final_df)

# %%
# 5 & 6. LOOKUPS AND RETURNS DATA
print("Fetching Lookups...")

df_lookup = get_sheet_df(client, LOOKUP_ID, "Sheet4")
lookup_map = dict(zip(df_lookup.iloc[:, 0].astype(str).str.lower().str.strip(), df_lookup.iloc[:, 1]))

print("Fetching Returns Data...")
df_returns_links = get_sheet_df(client, RETURNS_SALES_ID, "returns")
ret_url = [url for url in df_returns_links.iloc[:, 2] if "http" in url][-1]
df_ret = download_csv_tqdm(ret_url, desc="Downloading Returns CSV")

# STANDARD LOWERCASE CLEANING
df_ret['warehouse_name'] = df_ret['warehouse_name'].astype(str).str.strip().str.lower()
df_ret = df_ret[df_ret["pickup_status"].isin(['COMPLETED', 'PICKED', 'APPROVED'])]
df_ret['adjusted_time'] = pd.to_datetime(df_ret['pickup_unit_created_at'], errors='coerce')
df_ret['adjusted_time'] = df_ret['adjusted_time'] - pd.Timedelta(hours=5, minutes=30)
df_ret = df_ret[df_ret['adjusted_time'].dt.date == YESTERDAY]

df_ret['reason_clean'] = df_ret['reason'].astype(str).str.strip().str.upper()
df_ret['sub_reason_clean'] = df_ret['sub_reason'].astype(str).str.strip().str.upper()

# Rely solely on the 'vertical' column for mapping
df_ret['vertical_clean'] = df_ret['vertical'].astype(str).str.strip().str.lower()
df_ret['fsn_clean'] = df_ret['fsn'].astype(str).str.strip()

def classify_ret(row):
    vert = str(row.get('vertical_clean', '')).strip().lower()
    lkp = str(lookup_map.get(vert, '')).strip().lower()
    fsn = str(row.get('fsn_clean', '')).strip()
    if fsn in freshmilk_fsn_set: return 'Milk'
    if re.search(r'fruits|vegetables', vert): return 'FnV'
    if lkp == 'electronics': return 'Electronics'
    if lkp == 'daily use': return 'Daily use'
    return 'Other_Vert'

df_ret['cat'] = df_ret.apply(classify_ret, axis=1)

# NOTE: EXPIRED_PRODUCT sub-reason always maps to Expiry, regardless of category.
# DEFECTIVE_PRODUCT reason is grouped under Quality.
def classify_issue(row):
    r = str(row.get('reason_clean', '')).strip().upper()
    sr = str(row.get('sub_reason_clean', '')).strip().upper()
    cat = row.get('cat', '')
    if sr == 'EXPIRED_PRODUCT': return 'Expiry'
    if r == 'QUALITY_ISSUE': return 'Quality'
    if r == 'DAMAGED_PRODUCT': return 'Damage'
    if r == 'MISSING_ITEM': return 'Missing'
    if r == 'MISSHIPMENT': return 'Misshipment'
    if r == 'SUPPLY_CHAIN_ISSUES':
        if sr == 'NOT_DELIVERED_WRONGLY_MARKED_DELIVERED': return 'SCM - MDND'
        if sr == 'PICKUP_DONE_RETURN_CANCELLED': return 'SCM - PDRC'
        if sr == 'PRTO_DENIED_FOR_REMORSE': return 'SCM - pRTO D'
        return 'SCM'
    if r == 'DEFECTIVE_PRODUCT': return 'Quality'
    return 'Others'

df_ret['issue'] = df_ret.apply(classify_issue, axis=1)

ret_metrics = []
for store, grp in df_ret.groupby('warehouse_name'):
    metrics = {'store_id': store}
    metrics['Total unique complaints orders'] = grp['order_external_id'].nunique()
    metrics['Total unique complaints Return IDs'] = grp['return_id'].nunique()
    
    for issue_type in ['Quality', 'Damage', 'Expiry', 'Missing', 'Misshipment', 'SCM', 'SCM - MDND', 'SCM - PDRC', 'SCM - pRTO D', 'Others']:
        if issue_type == 'SCM':
            i_grp = grp[grp['issue'].isin(['SCM', 'SCM - MDND', 'SCM - PDRC', 'SCM - pRTO D'])]
        else:
            i_grp = grp[grp['issue'] == issue_type]
        metrics[f'Total unique Complaints Orders - {issue_type}'] = i_grp['order_external_id'].nunique()
        metrics[f'Total unique Complaints Return IDs - {issue_type}'] = i_grp['return_id'].nunique()
        
        if issue_type in ['Quality', 'Damage']:
            for cat in ['Milk', 'FnV', 'Electronics', 'Daily use']:
                c_grp = i_grp[i_grp['cat'] == cat]
                metrics[f'Total unique Complaints Orders - {issue_type} {cat}'] = c_grp['order_external_id'].nunique()
                metrics[f'Total unique Complaints Return IDs - {issue_type} {cat}'] = c_grp['return_id'].nunique()
                
    ret_metrics.append(metrics)
    
df_ret_agg = pd.DataFrame(ret_metrics)
if not df_ret_agg.empty:
    final_df = pd.merge(final_df, df_ret_agg, on='store_id', how='left')
print("Step 5 & 6 Complete.")

# %%
# 7. SKU SALES UNITS
print("Fetching SKU Sales units...")
df_sku_links = get_sheet_df(client, RETURNS_SALES_ID, "SKU_Order_id")
sku_url = [url for url in df_sku_links.iloc[:, 2] if "http" in url][-1]
df_sku = download_csv_tqdm(sku_url, desc="Downloading SKU Sales CSV")

# STANDARD LOWERCASE CLEANING
df_sku['warehouse_name'] = df_sku['warehouse_name'].astype(str).str.strip().str.lower()

df_sku['lookup_res'] = df_sku['vertical'].astype(str).str.strip().str.lower().map(lookup_map).str.lower()
df_sku['is_fnv'] = df_sku['vertical'].astype(str).str.contains('fruits|vegetables', flags=re.IGNORECASE, regex=True, na=False)

sku_agg = df_sku.groupby('warehouse_name').apply(lambda x: pd.Series({
    'FnV units count': x[x['is_fnv']]['product_fsn'].count(), 
    'Electronics units count': x[x['lookup_res'] == 'electronics']['product_fsn'].count()
})).reset_index().rename(columns={'warehouse_name': 'store_id'})

final_df = pd.merge(final_df, sku_agg, on='store_id', how='left')
print("Step 7 Complete.")
save_checkpoint(final_df)

# %%
# 8. STOREWISE SUMMARY & MILK JOINED
print("Fetching Milk & SDE Summaries...")
df_summ = get_sheet_df(client, STOREWISE_SUMMARY_ID, "StoreWise_Summary")
df_summ['date'] = pd.to_datetime(df_summ['date'], errors='coerce').dt.date
df_summ = df_summ[df_summ['date'] == YESTERDAY]

cols_wxy = df_summ.columns[22:25] 
df_summ['Milk Sales units - SDE'] = df_summ[cols_wxy].apply(lambda x: pd.to_numeric(x.astype(str).str.replace(',', '', regex=False), errors='coerce')).sum(axis=1)
df_summ = df_summ.rename(columns={'store_code': 'store_id', 'Total_Sales_Units': 'Milk sales units'})

# STANDARD LOWERCASE CLEANING
df_summ['store_id'] = df_summ['store_id'].astype(str).str.strip().str.lower()

# --- FIX: Clean numbers and GROUP BY store_id to prevent duplicate row explosions ---
df_summ['Milk sales units'] = pd.to_numeric(df_summ['Milk sales units'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
df_summ_agg = df_summ.groupby('store_id')[['Milk sales units', 'Milk Sales units - SDE']].sum().reset_index()

final_df = pd.merge(final_df, df_summ_agg, on='store_id', how='left')

df_mj = get_sheet_df(client, MILK_JOINED_ID, "MilkJoined")
df_mj['DateCol'] = pd.to_datetime(df_mj.iloc[:, 20], errors='coerce').dt.date
df_mj = df_mj[df_mj['DateCol'] == YESTERDAY]

df_mj_sde = df_mj[df_mj.iloc[:, 27].astype(str).str.strip().str.upper() == 'SDE'].copy()
store_col_mj = df_mj_sde.columns[2]

# STANDARD LOWERCASE CLEANING
df_mj_sde[store_col_mj] = df_mj_sde[store_col_mj].astype(str).str.strip().str.lower()

mj_agg = df_mj_sde.groupby(store_col_mj).size().reset_index(name='Milk Complaints units - SDE') 
mj_agg.rename(columns={store_col_mj: 'store_id'}, inplace=True)
final_df = pd.merge(final_df, mj_agg, on='store_id', how='left')
print("Step 8 Complete.")

# %%
# 9. INF DATA
print("Fetching INF Data...")
df_inf = get_sheet_df(client, INF_DATA_ID, tab_name="Analysis", header_row=1) 

if not df_inf.empty:
    df_inf = df_inf.iloc[:, 3:10].copy()
    df_inf.columns = [str(c).strip() for c in df_inf.columns]
    fc_col = next((c for c in df_inf.columns if c.lower() == 'fc'), None)
    if fc_col:
        df_inf.rename(columns={fc_col: 'store_id'}, inplace=True)
    
    # STANDARD LOWERCASE CLEANING
    df_inf['store_id'] = df_inf['store_id'].astype(str).str.strip().str.lower()
    
    df_inf = df_inf.loc[:, ~df_inf.columns.duplicated()].copy()
    final_df = pd.merge(final_df, df_inf.drop_duplicates(subset=['store_id']), on='store_id', how='left')

df_pivot_inf = get_sheet_df(client, PIVOT_INF_ID, "Pivot-INF")
df_pivot_inf['order_date'] = pd.to_datetime(df_pivot_inf.iloc[:, 11], errors='coerce').dt.date 
df_pivot_inf = df_pivot_inf[df_pivot_inf['order_date'] == YESTERDAY]

store_col_piv = df_pivot_inf.columns[0]
# STANDARD LOWERCASE CLEANING
df_pivot_inf[store_col_piv] = df_pivot_inf[store_col_piv].astype(str).str.strip().str.lower()

col_10_name = df_pivot_inf.columns[10]
df_pivot_inf[col_10_name] = pd.to_numeric(df_pivot_inf[col_10_name], errors='coerce')
p_inf_agg = df_pivot_inf.groupby(store_col_piv)[col_10_name].sum().reset_index() 
p_inf_agg.columns = ['store_id', 'inf_order_wise count']

final_df = pd.merge(final_df, p_inf_agg, on='store_id', how='left')
print("Step 9 Complete.")
save_checkpoint(final_df)

# %%
# 10. CHILLER CHECK & WAREHOUSE MAPPING
print("Fetching Warehouse Mapping...")

# 1. FETCH MAPPING DETAILS
df_mapping = get_sheet_df(client, STORE_MASTER_ID, "Mapping Details")

# Column F (Index 5) = Warehouse ID, Column I (Index 8) = Warehouse Name
wh_ids = df_mapping.iloc[:, 5].astype(str).str.strip().str.lower()
wh_names = df_mapping.iloc[:, 8].astype(str).str.strip()

# Build dictionaries for two-way translation
name_to_id_map = dict(zip(wh_names.str.lower(), wh_ids))
id_to_name_map = dict(zip(wh_ids, wh_names))

# 2. ADD WAREHOUSE NAME TO FINAL_DF
if 'Warehouse Name' not in final_df.columns:
    final_df['Warehouse Name'] = final_df['store_id'].map(id_to_name_map)
    print("Added 'Warehouse Name' column to final_df.")

# 3. PROCESS CHILLER DATA
print("Fetching Chiller Data...")
df_chiller = get_sheet_df(client, CHILLER_CHECK_ID, "Full_Vehicle_Scan")

if not df_chiller.empty:
    # Start from row 15,000 to the end
    df_chiller = df_chiller.iloc[15000:].copy()

    # Parse dates only on the remaining dataset
    df_chiller['DateCol'] = pd.to_datetime(df_chiller.iloc[:, 1], errors='coerce').dt.date 
    df_chiller = df_chiller[df_chiller['DateCol'] == YESTERDAY]

    def process_chiller(group):
        # Squeeze all double spaces/tabs into a single space, and lowercase it
        clean_category = group.iloc[:, 9].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip().str.lower()
        
        is_chiller_mask = (clean_category == 'chiller - non meat')
        is_chiller = any(is_chiller_mask) 
        
        # Grab temps only for the rows that match the chiller mask
        temps = group[is_chiller_mask].iloc[:, 10].tolist() 
        
        has_before_8 = any((pd.to_datetime(t, errors='coerce').hour < 8) for t in group.iloc[:, 1]) if not group.empty else False
        
        return pd.Series({
            'Chiller Non-Meat [Y/N]': 'Y' if is_chiller else 'N',
            'App-sheet temperature adherence?': temps[0] if temps else None,
            'App sheet filled before 8 AM?': 'Y' if has_before_8 else 'N'
        })

    if not df_chiller.empty:
        # --- FIX: Grab the full name, clean it, and translate to ID ---
        raw_chiller_names = df_chiller.iloc[:, 3].astype(str).str.strip().str.lower()
        df_chiller['store_id'] = raw_chiller_names.map(name_to_id_map)
        
        # Drop rows that couldn't find a matching ID
        df_chiller = df_chiller.dropna(subset=['store_id'])
        
        if not df_chiller.empty:
            chiller_agg = df_chiller.groupby('store_id').apply(process_chiller).reset_index()
            final_df = pd.merge(final_df, chiller_agg, on='store_id', how='left')
        else:
            print("Warning: None of the Chiller Warehouse Names matched the Mapping Details!")
    else:
        print(f"No chiller data matched the date: {YESTERDAY}")

print("Step 10 Complete.")

# %%
# 11. INDENT DATA
print("Fetching Indent Data...")

# 1. Actual OPD
df_act_opd = get_sheet_df(client, INDENT_ID, "Actual_OPD")
if not df_act_opd.empty:
    df_act_subset = df_act_opd.iloc[:, [0, 25]].copy()
    df_act_subset.columns = ['store_id', 'Actual Indent']
    
    df_act_subset['store_id'] = df_act_subset['store_id'].astype(str).str.strip().str.lower()
    # --- FIX: Strip commas just in case ---
    df_act_subset['Actual Indent'] = pd.to_numeric(df_act_subset['Actual Indent'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    
    act_agg = df_act_subset.groupby('store_id')['Actual Indent'].sum().reset_index()
    final_df = pd.merge(final_df, act_agg, on='store_id', how='left')

# 2. Planned OPD
df_plan_opd = get_sheet_df(client, INDENT_ID, "Planned_OPD")
if not df_plan_opd.empty:
    df_plan_subset = df_plan_opd.iloc[:, [5, 1]].copy()
    df_plan_subset.columns = ['store_id', 'Planned Indent']
    
    df_plan_subset['store_id'] = df_plan_subset['store_id'].astype(str).str.strip().str.lower()
    # --- FIX: Strip commas just in case ---
    df_plan_subset['Planned Indent'] = pd.to_numeric(df_plan_subset['Planned Indent'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    
    plan_agg = df_plan_subset.groupby('store_id')['Planned Indent'].sum().reset_index()
    final_df = pd.merge(final_df, plan_agg, on='store_id', how='left')

print("Step 11 Complete.")

# %%
# 12. MANUAL TTL SCRIPT LOGIC
print("Fetching Manual TTL Data...")
df_ttl_links = get_sheet_df(client, TTL_D_1, "Sheet1")
ttl_url = [url for url in df_ttl_links.iloc[-1] if str(url).startswith("http")][0]
df_ttl = download_csv_tqdm(ttl_url, desc="Downloading TTL CSV")

df_ttl.columns = df_ttl.columns.str.strip().str.lower()
df_ttl['created_date'] = pd.to_datetime(df_ttl['created_date'], errors='coerce').dt.date
df_ttl['status'] = df_ttl['status'].astype(str).str.strip().str.lower()
df_ttl['source_area'] = df_ttl['source_area'].astype(str).str.strip().str.lower()

df_ttl_filt = df_ttl[
    (df_ttl['status'] == 'completed') &
    (df_ttl['source_area'] == 'store') &
    (df_ttl['created_date'] == YESTERDAY)
].copy()

if not df_ttl_filt.empty:
    # STANDARD LOWERCASE CLEANING
    df_ttl_filt['mfc_id'] = df_ttl_filt['mfc_id'].astype(str).str.strip().str.lower()

    df_ttl_filt['transfer_list_type'] = df_ttl_filt['transfer_list_type'].astype(str).str.strip().str.lower()
    df_ttl_filt['categorization_vertical'] = df_ttl_filt['categorization_vertical'].astype(str).str.strip().str.lower()
    df_ttl_filt['picked_qty'] = pd.to_numeric(df_ttl_filt['picked_qty'], errors='coerce').fillna(0)
    df_ttl_filt['fsn'] = df_ttl_filt['fsn'].astype(str).str.strip()
    df_ttl_filt['is_milk'] = df_ttl_filt['fsn'].isin(freshmilk_fsn_set)
    
    c_at = pd.to_datetime(df_ttl_filt['created_at'], errors='coerce')
    comp_at = pd.to_datetime(df_ttl_filt['completed_at'], errors='coerce')
    df_ttl_filt['hrs_diff'] = (comp_at - c_at).dt.total_seconds() / 3600.0
    df_ttl_filt['non_promiseable_date'] = pd.to_datetime(df_ttl_filt['non_promiseable_date'], errors='coerce').dt.date

    df_ttl_filt['m3'] = df_ttl_filt['picked_qty'].where((df_ttl_filt['transfer_list_type'].isin(['jit_ttl', 'dst_ttl'])) & (df_ttl_filt['categorization_vertical'].isin(['fruits', 'vegetables'])), 0)
    df_ttl_filt['m4'] = df_ttl_filt['picked_qty'].where((df_ttl_filt['transfer_list_type'].isin(['jit_ttl', 'dst_ttl'])) & (df_ttl_filt['is_milk']), 0)
    df_ttl_filt['m5'] = df_ttl_filt['picked_qty'].where((df_ttl_filt['transfer_list_type'].isin(['jit_ttl', 'dst_ttl'])) & (~df_ttl_filt['categorization_vertical'].isin(['fruits', 'vegetables'])) & (~df_ttl_filt['is_milk']), 0)
    df_ttl_filt['m6'] = df_ttl_filt['picked_qty'].where(
    (df_ttl_filt['transfer_list_type'] == 'ems_ttl') & 
    (df_ttl_filt['hrs_diff'] < 4) & 
    (df_ttl_filt['is_milk']), 
    0
)
    df_ttl_filt['m7'] = df_ttl_filt['tl_created_qty'].where((df_ttl_filt['transfer_list_type'] == 'ems_ttl') & (df_ttl_filt['is_milk']), 0)
    df_ttl_filt['m8'] = df_ttl_filt['picked_qty'].where((df_ttl_filt['transfer_list_type'] == 'ems_ttl') & (df_ttl_filt['hrs_diff'] < 4) & (df_ttl_filt['categorization_vertical'].isin(['fruits', 'vegetables'])), 0)
    df_ttl_filt['m9'] = df_ttl_filt['tl_created_qty'].where((df_ttl_filt['transfer_list_type'] == 'ems_ttl') & (df_ttl_filt['categorization_vertical'].isin(['fruits', 'vegetables'])), 0)
    df_ttl_filt['m10'] = df_ttl_filt['picked_qty'].where((df_ttl_filt['transfer_list_type'].isin(['jit_ttl', 'dst_ttl'])) & (df_ttl_filt['is_milk']) & (df_ttl_filt['non_promiseable_date'] > YESTERDAY) & (df_ttl_filt['attribute_value'].isin(['EXPIRING_SOON', 'EXPIRED'])), 0)

    ttl_agg = df_ttl_filt.groupby('mfc_id').agg({
        'm3': 'sum', 'm4': 'sum', 'm5': 'sum', 'm6': 'sum',
        'm7': 'sum', 'm8': 'sum', 'm9': 'sum', 'm10': 'sum'
    }).reset_index().rename(columns={
        'mfc_id': 'store_id',
        'm3': 'Manual TTL qty - FnV',
        'm4': 'Manual TTL qty - Milk',
        'm5': 'Manual TTL qty - Others',
        'm6': 'Auto TTL qty Milk < 4 hrs completed',
        'm7': 'Auto TTL qty Milk - Generated',
        'm8': 'Auto TTL qty FnV < 4 hrs completed',
        'm9': 'Auto TTL qty FnV - Generated',
        'm10': 'Non-FEFO Manual TTL Milk qty'
    })
    final_df = pd.merge(final_df, ttl_agg, on='store_id', how='left')
print("Step 12 Complete.")
save_checkpoint(final_df)

# %%
# 13. LOCAL INVENTORY SNAPSHOT MERGE TO FINAL_DF
print("\n--- Starting Inventory Snapshot Update ---")

LOCAL_SNAPSHOT_PATH = r"C:\Users\muralimohana.s\Downloads\Copy of inventory_snapshot_2026-08-19_06.csv"

if not os.path.exists(LOCAL_SNAPSHOT_PATH):
    print(f"File not found: {LOCAL_SNAPSHOT_PATH}. Skipping snapshot merge.")
else:
    print(f"Fetching static FSN mapping from tab 'dictionary'...")
    mapping_ss = gspread_retry(lambda: client.open_by_key(MAPPING_SHEET_ID))
    mapping_sheet = mapping_ss.worksheet("dictionary")

    raw_mapping_rows = gspread_retry(lambda: mapping_sheet.get_all_values())
    mapping_headers = [str(h).strip() for h in raw_mapping_rows[0]]
    mapping_data = [dict(zip(mapping_headers, row)) for row in raw_mapping_rows[1:]]

    fsn_to_vertical = {str(row["ean_product_detail_fsn"]).strip(): str(row["analytic_vertical"]).strip() for row in mapping_data}
    
    final_aggregation = {}
    columns_to_keep = ['fsn', 'fc', 'atp','fcArea']

    for chunk in pd.read_csv(LOCAL_SNAPSHOT_PATH, usecols=columns_to_keep, chunksize=100000, low_memory=False):
        chunk.columns = [str(col).strip().lower() for col in chunk.columns]
        chunk['fsn'] = chunk['fsn'].astype(str).str.strip()
        chunk['atp'] = pd.to_numeric(chunk['atp'], errors='coerce').fillna(0)
        chunk = chunk[chunk['fcarea'].astype(str).str.strip().str.lower() == 'store']
        # STANDARD LOWERCASE CLEANING
        chunk['fc'] = chunk['fc'].astype(str).str.strip().str.lower()
        
        if chunk.empty:
            continue
            
        chunk['vertical'] = chunk['fsn'].map(fsn_to_vertical).fillna('Others')
        chunk['is_milk'] = chunk['fsn'].isin(freshmilk_fsn_set)
        chunk['Milk_Qty'] = chunk['atp'].where(chunk['is_milk'], 0)
        chunk['FnV_Qty'] = chunk['atp'].where(chunk['vertical'].isin(['Fruits', 'Vegetables']), 0)
        chunk['Others_Qty'] = chunk['atp'].where(~chunk['is_milk'] & ~chunk['vertical'].isin(['Fruits', 'Vegetables']), 0)
        
        chunk_summary = chunk.groupby('fc')[['Milk_Qty', 'FnV_Qty', 'Others_Qty']].sum().reset_index()
        
        for _, row in chunk_summary.iterrows():
            fc = row['fc']
            if fc not in final_aggregation:
                final_aggregation[fc] = {'Milk_Qty': 0, 'FnV_Qty': 0, 'Others_Qty': 0}
            final_aggregation[fc]['Milk_Qty'] += row['Milk_Qty']
            final_aggregation[fc]['FnV_Qty'] += row['FnV_Qty']
            final_aggregation[fc]['Others_Qty'] += row['Others_Qty']

    # Convert the aggregation dictionary to a DataFrame and merge it into final_df
    if final_aggregation:
        inv_df = pd.DataFrame.from_dict(final_aggregation, orient='index').reset_index()
        inv_df.rename(columns={'index': 'store_id'}, inplace=True)
        final_df = pd.merge(final_df, inv_df, on='store_id', how='left')
        print("✅ Inventory Snapshot metrics added to final_df.")
    else:
        print("No valid data found in snapshot to merge.")


# %%
# 14. 6AM COMBINED REPORT ETL MERGE TO FINAL_DF
print("\n--- Starting 6AM Combined Report ETL ---")

# Helpers for the ETL Engine
def find_headers_in_top_rows(worksheet, anchors, row_limit=10):
    all_rows = worksheet.get_values(f"A1:ZZ{row_limit}")
    for idx, row in enumerate(all_rows):
        normalized_row = [str(cell).strip().lower() for cell in row if cell]
        if any(a in normalized_row for a in anchors):
            return [str(cell).strip() for cell in row if cell], idx + 1
    return [h for h in worksheet.row_values(1) if h], 1

def resolve_col(actual_cols, norm_cols, label):
    key = label.strip().lower()
    if key in norm_cols:
        return actual_cols[norm_cols.index(key)]
    return None

def clean_series(frame, col):
    if col is None or col not in frame.columns:
        return None
    return frame[col].astype(str).str.strip()

def first_nonempty_map(frame, key_col, val_col):
    if val_col is None or val_col not in frame.columns:
        return {}
    t = frame[[key_col, val_col]].copy()
    t[val_col] = t[val_col].astype(str).str.strip()
    t = t[(t[val_col] != "") & (t[val_col].str.lower() != "nan")]
    if t.empty: return {}
    return t.groupby(key_col)[val_col].first().to_dict()

def sum_map(frame, key_col, val_col):
    if val_col is None or val_col not in frame.columns:
        return {}
    t = frame[[key_col, val_col]].copy()
    t['_n'] = pd.to_numeric(t[val_col].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    return t.groupby(key_col)['_n'].sum().to_dict()

def grn_minmax_maps(frame, key_col, grn_col):
    if grn_col is None or grn_col not in frame.columns:
        return {}, {}
    t = frame[[key_col, grn_col]].copy()
    t['_dt'] = pd.to_datetime(t[grn_col].astype(str).str.strip(), errors='coerce', dayfirst=False)
    t = t.dropna(subset=['_dt'])
    if t.empty: return {}, {}
    min_idx = t.groupby(key_col)['_dt'].idxmin()
    max_idx = t.groupby(key_col)['_dt'].idxmax()
    return t.loc[min_idx].set_index(key_col)[grn_col].astype(str).str.strip().to_dict(), \
           t.loc[max_idx].set_index(key_col)[grn_col].astype(str).str.strip().to_dict()

def find_header_row_raw(all_values, anchors, row_limit=10):
    for idx, row in enumerate(all_values[:row_limit]):
        norm = [str(c).strip().lower() for c in row]
        if any(a in norm for a in anchors):
            return idx
    return 0

def _to_num(x):
    try:
        s = str(x).replace(',', '').strip()
        if s == "" or s.lower() == "nan": return 0.0
        return float(s)
    except: return 0.0

def _num_out_safe(v):
    if v is None: return ""
    try: return int(v) if float(v).is_integer() else float(v)
    except: return v

# STANDARD LOWERCASE CLEANING
def norm_code(x):
    s = str(x).strip()
    if s == "" or s.lower() == "nan": return ""
    if s.endswith(".0") and s[:-2].isdigit(): s = s[:-2]
    return s.lower()

def summary_first_map(rows, sc_idx, val_idx):
    m = {}
    need = max(sc_idx, val_idx)
    for r in rows:
        if len(r) > need:
            sc = norm_code(r[sc_idx])
            if sc and sc not in m:
                v = str(r[val_idx]).strip()
                if v and v.lower() != 'nan':
                    m[sc] = v
    return m

def summary_sum_map(rows, sc_idx, val_idxs):
    m = {}
    need = max([sc_idx] + val_idxs)
    for r in rows:
        if len(r) > need:
            sc = norm_code(r[sc_idx])
            if sc:
                m[sc] = m.get(sc, 0.0) + sum(_to_num(r[vi]) for vi in val_idxs)
    return m

def parse_time_of_day(value):
    s = str(value).strip()
    if not s or s.lower() in ('nan', 'none', 'nat'): return None
    if '.' in s:
        try:
            f = float(s)
            frac = f - int(f)
            if frac < 0: frac += 1.0
            total = round(frac * 86400) % 86400
            h, rem = divmod(total, 3600)
            m, sec = divmod(rem, 60)
            return datetime(1900, 1, 1, h, m, sec).time()
        except: pass
    for fmt in ("%I:%M:%S %p", "%I:%M %p", "%I:%M:%S%p", "%I:%M%p", "%H:%M:%S", "%H:%M"):
        try: return datetime.strptime(s, fmt).time()
        except: continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ts = pd.to_datetime(s, errors='raise')
        return ts.to_pydatetime().time()
    except: return None

def format_arrival_with_date(value, invoice_date_str, fallback_date_str, unparsed_bucket=None):
    raw = str(value).strip()
    if not raw or raw.lower() == 'nan': return ""
    base = str(invoice_date_str).strip() or str(fallback_date_str).strip()
    try: inv_dt = datetime.strptime(base, "%Y-%m-%d")
    except: return raw
    t = parse_time_of_day(raw)
    if t is None:
        if unparsed_bucket is not None: unparsed_bucket.append(raw)
        return raw
    date_part = inv_dt - timedelta(days=1) if t.hour >= 18 else inv_dt
    canonical = datetime.combine(date_part.date(), t)
    return canonical.strftime("%Y-%m-%d %I:%M:%S %p")

# DYNAMIC DRIVE SEARCH FOR 6AM REPORT
report_name = f"6AM_combined_report_({YESTERDAY_STR})"
print(f"Searching Google Drive for: {report_name}...")

search_query = f"name contains '{report_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
search_results = drive_service.files().list(
    q=search_query, 
    fields="files(id, name)",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True
).execute()

found_files = search_results.get('files', [])

if not found_files:
    print(f"❌ Error: Could not find '{report_name}' in Google Drive. Skipping 6AM ETL.")
else:
    backup_file_id = found_files[0]['id']
    print(f"✅ Found File: {found_files[0]['name']} (ID: {backup_file_id})")
    
    try:
        bk_ss = gspread_retry(lambda: client.open_by_key(backup_file_id))
        bk_ws = None
        for candidate in ("Main_file", "Sheet1"):
            try:
                bk_ws = bk_ss.worksheet(candidate)
                break
            except gspread.WorksheetNotFound:
                continue
        if bk_ws is None:
            bk_ws = bk_ss.get_worksheet(0)

        bk_headers, bk_header_row_idx = find_headers_in_top_rows(bk_ws, ['warehouse id', 'invoice date', 'vertical', 'fsn'])
        raw_rows = bk_ws.get_all_values()[bk_header_row_idx:]
        
        if raw_rows:
            df = pd.DataFrame(raw_rows, columns=bk_headers)
            df.columns = [str(c).strip() for c in df.columns]
            actual_cols = list(df.columns)
            norm_cols = [c.lower() for c in actual_cols]

            SRC_LABELS = {
                'warehouse_id': 'Warehouse ID', 'invoice_date': 'Invoice Date', 'fnv_flag': 'FnV / DBEFM/ Others',
                'vertical': 'Vertical', 'fsn': 'FSN', 'fnv_store_in_time': 'FnV Store in Time',
                'dbefm_store_in_time': 'DBEFM Store in Time', 'grn_timestamp': 'grn_timestamp',
                'expected_qty': 'Expected Quantity', 'damaged_qty': 'Damaged Quantity',
            }
            col = {k: resolve_col(actual_cols, norm_cols, lbl) for k, lbl in SRC_LABELS.items()}
            
            wh_col = col['warehouse_id']
            df[wh_col] = df[wh_col].astype(str).str.strip().str.lower()
            df = df[(df[wh_col] != "") & (df[wh_col] != "nan")]
            
            unique_store_codes = sorted(df[wh_col].unique().tolist())
            
            fnv_flag_s = clean_series(df, col['fnv_flag'])
            df_fnv = df[fnv_flag_s.str.lower() == 'fnv'].copy() if fnv_flag_s is not None else df.iloc[0:0].copy()
            
            fsn_s = clean_series(df, col['fsn'])
            df_milk = df[fsn_s.isin(freshmilk_fsn_set)].copy() if fsn_s is not None else df.iloc[0:0].copy()

            date_map = first_nonempty_map(df, wh_col, col['invoice_date'])
            
            fnv_arrival_map = first_nonempty_map(df_fnv, wh_col, col['fnv_store_in_time'])
            fnv_min_map, fnv_max_map = grn_minmax_maps(df_fnv, wh_col, col['grn_timestamp'])
            
            milk_arrival_map = first_nonempty_map(df_milk, wh_col, col['dbefm_store_in_time'])
            milk_min_map, milk_max_map = grn_minmax_maps(df_milk, wh_col, col['grn_timestamp'])
            
            fnv_exp_map = sum_map(df_fnv, wh_col, col['expected_qty'])
            fnv_dmg_map = sum_map(df_fnv, wh_col, col['damaged_qty'])
            milk_exp_map = sum_map(df_milk, wh_col, col['expected_qty'])
            milk_dmg_map = sum_map(df_milk, wh_col, col['damaged_qty'])
            
            SUMM_COL = {'store_code': 1, 'fnv_vehicle_on_time': 3, 'dbefm_vehicle_on_time': 39, 'fnv_total_a': 7, 'fnv_total_b': 9, 'fnv_putaway': 8, 'dbefm_total_a': 43, 'dbefm_total_b': 45, 'dbefm_putaway': 44}
            fnv_vot_map, dbefm_vot_map, fnv_total_map, fnv_putaway_map, dbefm_total_map, dbefm_putaway_map = {}, {}, {}, {}, {}, {}

            try:
                summ_ws = bk_ss.worksheet("Overall Summary_Store_wise")
                summ_all = summ_ws.get_all_values()
                summ_hdr_idx = find_header_row_raw(summ_all, ['warehouse id', 'store code'])
                summ_rows = summ_all[summ_hdr_idx + 1:]
                
                sc_i = SUMM_COL['store_code']
                fnv_vot_map = summary_first_map(summ_rows, sc_i, SUMM_COL['fnv_vehicle_on_time'])
                dbefm_vot_map = summary_first_map(summ_rows, sc_i, SUMM_COL['dbefm_vehicle_on_time'])
                fnv_total_map = summary_sum_map(summ_rows, sc_i, [SUMM_COL['fnv_total_a'], SUMM_COL['fnv_total_b']])
                dbefm_total_map = summary_sum_map(summ_rows, sc_i, [SUMM_COL['dbefm_total_a'], SUMM_COL['dbefm_total_b']])
                fnv_putaway_map = summary_sum_map(summ_rows, sc_i, [SUMM_COL['fnv_putaway']])
                dbefm_putaway_map = summary_sum_map(summ_rows, sc_i, [SUMM_COL['dbefm_putaway']])
            except gspread.WorksheetNotFound:
                pass

            # Build a DataFrame dictionary from the results
            six_am_data = []
            unparsed_times = []
            for sc in unique_store_codes:
                inv_date_val = str(date_map.get(sc, YESTERDAY_STR)) or YESTERDAY_STR
                nk = norm_code(sc)
                
                row_dict = {
                    'store_id': sc,
                    'Vehicle Arrival Time - FnV': format_arrival_with_date(fnv_arrival_map.get(sc, ""), inv_date_val, YESTERDAY_STR, unparsed_times),
                    'Unload Start Time - FnV': fnv_min_map.get(sc, ""),
                    'Unload End Time - FnV': fnv_max_map.get(sc, ""),
                    'Vehicle Arrival Time - Milk': format_arrival_with_date(milk_arrival_map.get(sc, ""), inv_date_val, YESTERDAY_STR, unparsed_times),
                    'Unload Start Time - Milk': milk_min_map.get(sc, ""),
                    'Unload End Time - Milk': milk_max_map.get(sc, ""),
                    'I/B Dispatched Qty FnV': int(fnv_exp_map.get(sc, 0)),
                    'I/B Rejected Qty FnV': int(fnv_dmg_map.get(sc, 0)),
                    'I/B Dispatched Qty Milk': int(milk_exp_map.get(sc, 0)),
                    'I/B Rejected Qty Milk': int(milk_dmg_map.get(sc, 0)),
                    'FnV Vehicle on time?': fnv_vot_map.get(nk, ""),
                    'DBEFM Vehicle on time?': dbefm_vot_map.get(nk, ""),
                    'Total expected qty - FnV': _num_out_safe(fnv_total_map.get(nk)),
                    'Putaway on time qty - FnV': _num_out_safe(fnv_putaway_map.get(nk)),
                    'Total expected qty - DBEFM': _num_out_safe(dbefm_total_map.get(nk)),
                    'Putaway on time qty - DBEFM': _num_out_safe(dbefm_putaway_map.get(nk))
                }
                six_am_data.append(row_dict)
            
            if six_am_data:
                df_6am = pd.DataFrame(six_am_data)
                df_6am = df_6am.drop_duplicates(subset=['store_id'])
                final_df = pd.merge(final_df, df_6am, on='store_id', how='left')
                print("✅ 6AM Combined Report metrics added to final_df.")

    except Exception as critical_err:
        print(f"\nCRITICAL CORE EXCEPTION: Pipeline terminated unexpectedly: {critical_err}")
        traceback.print_exc()

# %%
# 15. APPENDING TO MASTER GOOGLE SHEET (FINAL UPLOAD)
print(f"\nFormatting and Appending data to '{DEST_TAB_NAME}' Sheet...")

# Clean NaNs before uploading
final_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Convert the dataframe to 'object' type so it accepts both numbers and empty strings
final_df = final_df.astype(object).fillna("")

# Rename standard identifier columns to match your destination sheet
final_df.rename(columns={'store_id': 'Store Code', 'StoresZone': 'Zone', 'City.Final': 'City'}, inplace=True)

# Filter out empty Store Codes BEFORE aligning with Google Sheet headers
if 'Store Code' in final_df.columns:
    final_df = final_df[final_df['Store Code'] != ""]

dest_ss = gspread_retry(lambda: client.open_by_key(DEST_SPREADSHEET_ID))
dest_ws = dest_ss.worksheet(DEST_TAB_NAME)

all_values = gspread_retry(lambda: dest_ws.get_all_values())

if not all_values:
    print("The destination sheet is completely empty. Creating headers automatically...")
    existing_headers = list(final_df.columns)
    gspread_retry(lambda: dest_ws.append_row(existing_headers, value_input_option="USER_ENTERED"))
else:
    # Pull existing Headers from Row 1
    existing_headers = all_values[0]
    
    # --- NEW FEATURE: Detect and add new columns ---
    new_cols = [col for col in final_df.columns if col not in existing_headers]
    if new_cols:
        print(f"Detected {len(new_cols)} new column(s): {new_cols}")
        print("Expanding Google Sheet headers...")
        existing_headers.extend(new_cols)
        
        # Update Row 1 with the expanded header list
        def _update_headers():
            try:
                dest_ws.update(values=[existing_headers], range_name="A1")
            except TypeError:
                dest_ws.update("A1", [existing_headers])
        gspread_retry(_update_headers)

# Ensure all target columns exist in our df, if not add them empty
for h in existing_headers:
    if h not in final_df.columns:
        final_df[h] = ""
        
# Reorder DataFrame to align perfectly with the destination sheet headers
upload_df = final_df[existing_headers]

if not upload_df.empty:
    upload_data = upload_df.values.tolist()
    
    print(f"Appending {len(upload_data)} rows to {DEST_SPREADSHEET_ID} -> '{DEST_TAB_NAME}' tab.")
    CHUNK_SIZE = 500
    for start in range(0, len(upload_data), CHUNK_SIZE):
        chunk = upload_data[start:start + CHUNK_SIZE]
        gspread_retry(lambda chunk=chunk: dest_ws.append_rows(chunk, value_input_option="USER_ENTERED"))
        print(f"  Uploaded rows {start+1}-{start+len(chunk)} of {len(upload_data)}")
    print("Done! Mission Accomplished.")
else:
    print("No data compiled to upload for yesterday.")


Imports loaded.
--- Pipeline Ready for D-1 Date: 2026-08-19 ---
Fetching Fresh Milk FSN mapping...
Fetching Store Details...
Step 1 Complete.
  💾 Checkpoint saved → C:\Users\muralimohana.s\Downloads\pipeline_checkpoint.pkl
Fetching Event Hub Data...
Step 2 Complete.
Fetching Store Dashboard...
Step 3 Complete.
Fetching TAT Data...


Step 4 Complete.
  💾 Checkpoint saved → C:\Users\muralimohana.s\Downloads\pipeline_checkpoint.pkl
Fetching Lookups...
Fetching Returns Data...


Step 5 & 6 Complete.
Fetching SKU Sales units...


Step 7 Complete.
  💾 Checkpoint saved → C:\Users\muralimohana.s\Downloads\pipeline_checkpoint.pkl
Fetching Milk & SDE Summaries...
Step 8 Complete.
Fetching INF Data...
Step 9 Complete.
  💾 Checkpoint saved → C:\Users\muralimohana.s\Downloads\pipeline_checkpoint.pkl
Fetching Warehouse Mapping...
Added 'Warehouse Name' column to final_df.
Fetching Chiller Data...
  ⚠️ Google API 503 — retrying in 15s (attempt 1/5)...
  ⚠️ Google API 429 — retrying in 30s (attempt 2/5)...
  ⚠️ Google API 429 — retrying in 60s (attempt 3/5)...
  ⚠️ Google API 429 — retrying in 120s (attempt 4/5)...


In [ ]:
# # %% [markdown]
# # # 16. RETURNS-ONLY REFRESH (multi-day, from local CSVs) -> CSV FOR MANUAL REVIEW
# # Fill in RETURNS_FILE_PATHS below with the returns export CSVs you want bucketed
# # (e.g. the daily exports from the 13th onwards). Each file's own dates are used -
# # no date filter is applied, so mix as many days/files as you like. Same
# # Quality/Damage/SCM/Expiry x Milk/FnV/Electronics/Daily use/Others logic as the
# # main pipeline's Step 5 & 6, one output row per store per day. Writes a CSV
# # instead of touching the Google Sheet - review and upload manually.

# # %%
# import re
# import pandas as pd
# import gspread
# from oauth2client.service_account import ServiceAccountCredentials

# JSON_PATH = r"C:\Users\muralimohana.s\Desktop\milk_npd\minutesreports-8b030d50afd3.json"
# SCOPES = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
# LOOKUP_ID = "10JYZdk4WGLvLIJborLPt0d3i3q0Bzo85CzD0hvD6L_A"
# MAPPING_SHEET_ID = "10JYZdk4WGLvLIJborLPt0d3i3q0Bzo85CzD0hvD6L_A"

# # --- Fill in the returns export CSV file paths you want bucketed (one per day) ---
# RETURNS_FILE_PATHS = [
#     r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_0a53284c7d5d8f29f5fa8c0016c67b631784406696899.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_2fcfa6cc4e55d7c3131b6319bb16f0db1784936703788.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_8bad0c54f9adee3c5e9647bdd8138ca21785109501569.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_11fc96590448db7a26af008e265d3ce41784245501983.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_56e946171c6078c52d7929845ddd38ca1783986301829 (1).csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_69de19c80a3bb8b7e1256c79ee027a831784159101865.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_89d89fb89b3f70c209035cc3f4f5edc81784850302034.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_364961e1e96e4c7747fc3aa6c82d260f1784677500753.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_a1fdd1deebfbb1710faabaa0d2be847f1785282304646.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_a4aa8ef03b85f21dd9360c5cec72c3301785368703996.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_b3c99a3d3e2920f526fa07059d8266091784072704537.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_c048d6c85ba9e302893833a013826f2e1784763903693.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_d1720562407d52f520e88953eb5b08871785023106155.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_dabf2d95c86b5704ab4c063b381998cd1784331907568.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_e94c3ba188faf083e4afd9112cdee8db1785195901272.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_fe94ba3996183bb56e12546c0bf197141784591106193.csv",
# r"C:\Users\muralimohana.s\Desktop\ret_13-17\superbi_BQ_BATCH_V2_P1_SUBS_c86ca0eaf4f42f6bdad7a9d98ed0313e1785455109805.csv"
# ]

# OUTPUT_CSV_PATH = r"C:\Users\muralimohana.s\Downloads\returns_only1.csv"

# def auth_gspread():
#     creds = ServiceAccountCredentials.from_json_keyfile_name(JSON_PATH, SCOPES)
#     return gspread.authorize(creds)

# client = auth_gspread()

# freshmilk_ws = client.open_by_key(MAPPING_SHEET_ID).worksheet("freshmilk_fsn")
# freshmilk_fsn_set = {str(v).strip() for v in freshmilk_ws.col_values(2)[1:] if str(v).strip()}

# lookup_ws = client.open_by_key(LOOKUP_ID).worksheet("Sheet4")
# lookup_rows = lookup_ws.get_all_values()[1:]
# lookup_map = {str(r[0]).strip().lower(): r[1] for r in lookup_rows if r}

# def classify_ret(row):
#     vert = str(row.get('vertical_clean', '')).strip().lower()
#     lkp = str(lookup_map.get(vert, '')).strip().lower()
#     fsn = str(row.get('fsn_clean', '')).strip()
#     if fsn in freshmilk_fsn_set: return 'Milk'
#     if re.search(r'fruits|vegetables', vert): return 'FnV'
#     if lkp == 'electronics': return 'Electronics'
#     if lkp == 'daily use': return 'Daily use'
#     return 'Other_Vert'

# def classify_issue(row):
#     r = str(row.get('reason_clean', '')).strip().upper()
#     sr = str(row.get('sub_reason_clean', '')).strip().upper()
#     cat = row.get('cat', '')
#     if sr == 'EXPIRED_PRODUCT': return 'Expiry'
#     if r == 'QUALITY_ISSUE': return 'Quality'
#     if r == 'DAMAGED_PRODUCT': return 'Damage'
#     if r == 'MISSING_ITEM': return 'Missing'
#     if r == 'MISSHIPMENT': return 'Misshipment'
#     if r == 'SUPPLY_CHAIN_ISSUES':
#         if sr == 'NOT_DELIVERED_WRONGLY_MARKED_DELIVERED': return 'SCM - MDND'
#         if sr == 'PICKUP_DONE_RETURN_CANCELLED': return 'SCM - PDRC'
#         if sr == 'PRTO_DENIED_FOR_REMORSE': return 'SCM - pRTO D'
#         return 'SCM'
#     if r == 'DEFECTIVE_PRODUCT': return 'Quality'
#     return 'Others'

# all_frames = []
# for path in RETURNS_FILE_PATHS:
#     print(f"Reading {path}...")
#     df_h = pd.read_csv(path, low_memory=False)
#     df_h['warehouse_name'] = df_h['warehouse_name'].astype(str).str.strip().str.lower()
#     df_h = df_h[df_h['pickup_status'].isin(['COMPLETED', 'PICKED', 'APPROVED'])]
#     df_h['adjusted_time'] = pd.to_datetime(df_h['pickup_unit_created_at'], errors='coerce') - pd.Timedelta(hours=5, minutes=30)
#     df_h['ret_date'] = df_h['adjusted_time'].dt.date
#     df_h['reason_clean'] = df_h['reason'].astype(str).str.strip().str.upper()
#     df_h['sub_reason_clean'] = df_h['sub_reason'].astype(str).str.strip().str.upper()
#     df_h['vertical_clean'] = df_h['vertical'].astype(str).str.strip().str.lower()
#     df_h['fsn_clean'] = df_h['fsn'].astype(str).str.strip()
#     all_frames.append(df_h)

# df_hist = pd.concat(all_frames, ignore_index=True)
# df_hist['cat'] = df_hist.apply(classify_ret, axis=1)
# df_hist['issue'] = df_hist.apply(classify_issue, axis=1)
# print("Loaded & classified returns data.")

# # %%
# # AGGREGATE PER STORE, PER DATE -> CSV FOR MANUAL REVIEW
# hist_metrics = []
# for (ret_date, store), grp in df_hist.groupby(['ret_date', 'warehouse_name']):
#     metrics = {'Date': ret_date, 'store_id': store}
#     metrics['Total unique complaints orders'] = grp['order_external_id'].nunique()
#     metrics['Total unique complaints Return IDs'] = grp['return_id'].nunique()

#     for issue_type in ['Quality', 'Damage', 'Expiry', 'Missing', 'Misshipment', 'SCM', 'SCM - MDND', 'SCM - PDRC', 'SCM - pRTO D', 'Others']:
#         if issue_type == 'SCM':
#             i_grp = grp[grp['issue'].isin(['SCM', 'SCM - MDND', 'SCM - PDRC', 'SCM - pRTO D'])]
#         else:
#             i_grp = grp[grp['issue'] == issue_type]
#         metrics[f'Total unique Complaints Orders - {issue_type}'] = i_grp['order_external_id'].nunique()
#         metrics[f'Total unique Complaints Return IDs - {issue_type}'] = i_grp['return_id'].nunique()

#         if issue_type in ['Quality', 'Damage']:
#             for cat in ['Milk', 'FnV', 'Electronics', 'Daily use']:
#                 c_grp = i_grp[i_grp['cat'] == cat]
#                 metrics[f'Total unique Complaints Orders - {issue_type} {cat}'] = c_grp['order_external_id'].nunique()
#                 metrics[f'Total unique Complaints Return IDs - {issue_type} {cat}'] = c_grp['return_id'].nunique()

#     hist_metrics.append(metrics)

# df_hist_agg = pd.DataFrame(hist_metrics).sort_values(['Date', 'store_id'])
# df_hist_agg.to_csv(OUTPUT_CSV_PATH, index=False)
# print(f"Saved {len(df_hist_agg)} rows to {OUTPUT_CSV_PATH}")

In [ ]:
# LOCAL_SNAPSHOT_PATH = r"C:\Users\muralimohana.s\Downloads\inventory_snapshot_2026-07-19_06.csv"
# OUTPUT_CSV_PATH = r"C:\Users\muralimohana.s\Downloads\fc_inventory_summary_store_19.csv"

# print(f"Fetching static FSN mapping from tab 'dictionary'...")
# mapping_ss = client.open_by_key(MAPPING_SHEET_ID)
# mapping_sheet = mapping_ss.worksheet("dictionary")

# raw_mapping_rows = mapping_sheet.get_all_values()
# mapping_headers = [str(h).strip() for h in raw_mapping_rows[0]]
# mapping_data = [dict(zip(mapping_headers, row)) for row in raw_mapping_rows[1:]]

# fsn_to_vertical = {str(row["ean_product_detail_fsn"]).strip(): str(row["analytic_vertical"]).strip() for row in mapping_data}

# final_aggregation = {}
# columns_to_keep = ['fsn', 'fc', 'atp', 'fcArea']

# for chunk in pd.read_csv(LOCAL_SNAPSHOT_PATH, usecols=columns_to_keep, chunksize=100000, low_memory=False):
#     chunk.columns = [str(col).strip().lower() for col in chunk.columns]
#     chunk['fsn'] = chunk['fsn'].astype(str).str.strip()
#     chunk['atp'] = pd.to_numeric(chunk['atp'], errors='coerce').fillna(0)
#     chunk['fc'] = chunk['fc'].astype(str).str.strip().str.lower()
#     chunk = chunk[chunk['fcarea'].astype(str).str.strip().str.lower() == 'store']
    

#     if chunk.empty:
#         continue

#     chunk['vertical'] = chunk['fsn'].map(fsn_to_vertical).fillna('Others')
#     chunk['is_milk'] = chunk['fsn'].isin(freshmilk_fsn_set)
#     chunk['Milk_Qty'] = chunk['atp'].where(chunk['is_milk'], 0)
#     chunk['FnV_Qty'] = chunk['atp'].where(chunk['vertical'].isin(['Fruits', 'Vegetables']), 0)
#     chunk['Others_Qty'] = chunk['atp'].where(~chunk['is_milk'] & ~chunk['vertical'].isin(['Fruits', 'Vegetables']), 0)

#     chunk_summary = chunk.groupby('fc')[['Milk_Qty', 'FnV_Qty', 'Others_Qty']].sum().reset_index()

#     for _, row in chunk_summary.iterrows():
#         fc = row['fc']
#         if fc not in final_aggregation:
#             final_aggregation[fc] = {'Milk_Qty': 0, 'FnV_Qty': 0, 'Others_Qty': 0}
#         final_aggregation[fc]['Milk_Qty'] += row['Milk_Qty']
#         final_aggregation[fc]['FnV_Qty'] += row['FnV_Qty']
#         final_aggregation[fc]['Others_Qty'] += row['Others_Qty']

# if final_aggregation:
#     inv_df = pd.DataFrame.from_dict(final_aggregation, orient='index').reset_index()
#     inv_df.rename(columns={'index': 'store_id'}, inplace=True)
#     inv_df.to_csv(OUTPUT_CSV_PATH, index=False)
#     print(f"Saved {len(inv_df)} rows to {OUTPUT_CSV_PATH}")
# else:
#     print("No valid data found in snapshot after fcarea='store' filter.")